In [1]:
import anndata
import numpy as np
import scvelo as scv
import scanpy as sc
import sys
import torch
import os.path
import deepvelo as dv
from deepvelo.utils.preprocess import autoset_coeff_s
from deepvelo.utils import (
    velocity,
    velocity_confidence,
    latent_time,
    update_dict,
    save_model_and_config,
    cross_boundary_correctness,
)
from deepvelo.utils.plot import statplot, compare_plot
from deepvelo import train, Constants
import pickle as pickle
import matplotlib.pyplot as plt
import pandas as pd
import unitvelo as utv
from os.path import exists
import time
method = 'Deepvelo2024'

Using backend: pytorch
2024-12-16 05:50:35.548710: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-16 05:50:35.549785: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-12-16 05:50:35.554113: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-12-16 05:50:35.568306: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-16 05:50:35.590951: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN whe

(Running UniTVelo 0.2.5.2)
2024-12-16 10:50:37


/data/luoya/env/Deepvelo-2024/lib/python3.9/site-packages/deepvelo/utils/plot.py:5: DeprecationWarning: deepvelo.utils.plot is deprecated. Please use deepvelo.plot.plot instead.
  warnings.warn(


In [ ]:
#dataset name
datasets=['Pancreas','DentateGyrus','Erythroid_Maturation','HumanBoneMarrow','Intestinal_organoid','mouse_retina','Hindbrain_GABA_Glio','organogenesis_chondrocyte']
#data path
data_dir = '/data/'
#result path
save_dir = '/result/'

In [ ]:
df_CB= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_CB)
df_IC= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_IC)

Empty DataFrame
Columns: [Mean, Time(s)]
Index: []
Empty DataFrame
Columns: [Mean, Time(s)]
Index: []


In [ ]:
for dataset in datasets:
    print(dataset)
    adata = sc.read_h5ad(data_dir + dataset +'/'+ f'{dataset}.h5ad')
    # specific configs to overide the default configs
    configs = {
        "name": "DeepVelo", # name of the experiment
        "loss": {"args": {"coeff_s": autoset_coeff_s(adata)}},
        "trainer": {"verbosity": 0}, # increase verbosity to show training progress
    }
    configs = update_dict(Constants.default_configs, configs)   
    start = time.time()
    scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=3000)
    scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
    trainer = dv.train(adata, dv.Constants.default_configs)
    scv.pp.neighbors(adata)
    scv.tl.velocity_graph(adata, vkey = 'velocity')
    scv.tl.velocity_embedding(adata, vkey = 'velocity')
    end = time.time()
    fix, ax = plt.subplots(1, 1, figsize = (8, 6))
    scv.pl.velocity_embedding_stream(adata, basis='umap', save = False, vkey='velocity',color="clusters",
                                     show = False, ax = ax)
    plt.savefig(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_{method}.svg')
    # Calculate performance metrics:
    file = open(data_dir + dataset +'/'+f'{dataset}_groundTruth.pickle' ,'rb')
    ground_truth = pickle.load(file)
    metrics = utv.evaluate(adata, ground_truth, 'clusters', 'velocity')
    if exists(save_dir + method +'/'+ '_CBDir_scores.csv'):
        tab = pd.read_csv(save_dir + method +'/'+'_CBDir_scores.csv', index_col = 0)
    else:
        tab_cb = pd.DataFrame(columns = list(metrics['Cross-Boundary Direction Correctness (A->B)'].keys())  + ['Mean', 'Time(s)'],
                 index = [dataset])
        tab_IC = pd.DataFrame(columns = list(metrics['In-cluster Coherence'].keys())  + ['Mean', 'Time(s)'],
                 index = [dataset])
    ##CBDir_scores
    cb_score = [np.mean(metrics['Cross-Boundary Direction Correctness (A->B)'][x])
                for x in metrics['Cross-Boundary Direction Correctness (A->B)'].keys()]
    tab_cb.loc[dataset,:] = cb_score + [np.mean(cb_score), end-start]
    df_CB=df_CB.append(pd.DataFrame([[np.mean(cb_score), end-start]],columns=df_CB.columns,index=[dataset]))
    tab_cb.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_CBDir_scores.csv')
    ##ICCoh_scores
    IC_score = [np.mean(metrics['In-cluster Coherence'][x])
                for x in metrics['In-cluster Coherence'].keys()]
    tab_IC.loc[dataset,:] = IC_score + [np.mean(IC_score), end-start]
    df_IC=df_IC.append(pd.DataFrame([[np.mean(IC_score), end-start]],columns=df_IC.columns,index=[dataset]))
    tab_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_ICCoh_scores.csv')
    
    adata.write_h5ad(save_dir + method +'/' +'CB_IC/'+ f'{dataset}_AnnData_Forscore.h5ad')

In [8]:
df_CB.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_CBDir_scores.csv')
df_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_ICCoh_scores.csv')